den store fil 

## vi har nu:

# Preprocessing
import af alle predictors
averaging til én værdi per år
kombination baseret på valgåret

# modelling
en simpel OLS model - med flere sæt predictors

# evaluation
En LOOCV



### vi mangler:
# Preprocessing
(en mere sofistikeret vægtning når flere  målinger pr år)
interpolation af manglende år
kombination baseret på alle år, evt baseret på vægte estimeret ved regression (eller en model der kan have en højere resolution i x end i y)

# Modelling
En analyse af colinearity/kombination af variable for at komme af med den
en ols model rafineret efter chapt 6
en form for bayesian regression eller noget?
En form for tree based method

# Evaluation
En leave future out validation (der er mange valg her fx ift om man kun skal evaluere på nogle få valg og om man skal fjerne det seenste)

en analyse af hvilke predictors der er vigtige og hvad det  btyder

In [95]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
import sklearn

In [96]:
import os

In [97]:
os.getcwd()

'c:\\Users\\carle\\Documents\\Cognitive_Science\\MASTER\\Data science\\Data science exam'

In [98]:
gini = pd.read_csv('data/gini.csv', index_col=0, skiprows=3)
gdp = pd.read_csv('data/gdp.csv', index_col=0, skiprows=3)
inflation = pd.read_csv('data/inflation.csv', index_col=0, skiprows=3)
unemployment = pd.read_csv('data/unemployment.csv', index_col=0, skiprows=3)

In [99]:
gdp.head()

,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
Country Name,,,,,,,,,,,,,,,,,,,,,
Aruba,ABW,GDP per capita growth (annual %),NY.GDP.PCAP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.485815,3.048518,0.951664,-23.466273,15.675519,11.038520,7.657640,6.181751,NaN,NaN
Africa Eastern and Southern,AFE,GDP per capita growth (annual %),NY.GDP.PCAP.KD.ZG,NaN,-2.182395,5.075379,2.795642,1.790384,2.225029,1.922442,...,0.000238,-0.064991,-0.709407,-5.405934,1.844404,1.068005,-0.604582,0.253854,NaN,NaN
Afghanistan,AFG,GDP per capita growth (annual %),NY.GDP.PCAP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.195570,-1.713743,0.856295,-5.382515,-22.584482,-7.576669,0.106093,NaN,NaN,NaN
Africa Western and Central,AFW,GDP per capita growth (annual %),NY.GDP.PCAP.KD.ZG,NaN,-0.251185,1.538542,4.740660,3.086962,1.823450,-3.703529,...,-0.355283,0.330202,0.792058,-6.003416,0.154018,2.064401,1.218110,2.107058,NaN,NaN
Angola,AGO,GDP per capita growth (annual %),NY.GDP.PCAP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-3.622865,-3.968525,-3.529000,-7.127615,-1.094341,0.991407,-1.808703,1.292933,NaN,NaN


In [100]:
gini, gdp, inflation, unemployment = [
    df.loc[['United States']]
    for df in [gini, gdp, inflation, unemployment]
]

In [101]:
gini, gdp, inflation, unemployment = [
    df.drop(columns=["Country Code", "Indicator Name", "Indicator Code", "Unnamed: 70"])
    for df in [gini, gdp, inflation, unemployment]
]

In [102]:
gini, gdp, inflation, unemployment = [
    df.melt(var_name="year", value_name=name)
    for df, name in zip(
        [gini, gdp, inflation, unemployment],
        ["gini", "gdp", "inflation", "unemployment"]
    )
]

In [103]:
gini['year'], gdp['year'], inflation['year'], unemployment['year'] = [
    df["year"].astype(int)
    for df in [gini, gdp, inflation, unemployment]
]

In [104]:
gdp

,year,gdp
0,1960,NaN
1,1961,0.618121
2,1962,4.480669
3,1963,2.908272
4,1964,4.340549
...,...,...
61,2021,5.888341
62,2022,1.923862
63,2023,2.035600
64,2024,1.794192


In [105]:
income = pd.read_csv('data/Real wage.csv')
satisfaction = pd.read_csv('data/satisfaction.csv')

In [106]:
income['median_real_wage'] = income['LES1252881600Q']

In [107]:
income['year'] = income['observation_date'].str[:4]
income.head()

,observation_date,LES1252881600Q,median_real_wage,year
0,1979-01-01,335.0,335.0,1979
1,1979-04-01,335.0,335.0,1979
2,1979-07-01,330.0,330.0,1979
3,1979-10-01,326.0,326.0,1979
4,1980-01-01,321.0,321.0,1980


In [108]:
satisfaction['year'] = satisfaction['In general, are you satisfied or dissatisfied with the way things are going in [your personal life/the United States] at this time? <br><b>% Satisfied</b>'].str[-4:]

In [109]:
yearly_wage = income.groupby('year')['median_real_wage'].mean()
yearly_pers = satisfaction.groupby('year')['Personal life'].mean()
yearly_us = satisfaction.groupby('year')['United States'].mean()

In [110]:
yearly_wage = pd.DataFrame(index= yearly_wage.index, data=yearly_wage.values, columns=['wage'])
yearly_pers = pd.DataFrame(index= yearly_pers.index, data=yearly_pers.values, columns=['pers'])
yearly_us = pd.DataFrame(index= yearly_us.index, data=yearly_us.values, columns=['us'])

In [111]:
yearly_wage.reset_index(inplace=True)
yearly_pers.reset_index(inplace=True)
yearly_us.reset_index(inplace=True)


In [112]:
yearly_pers.head()

,year,pers
0,1979,76.333333
1,1981,81.000000
2,1982,75.500000
3,1983,77.000000
4,1984,79.000000


In [113]:
election_results = pd.read_csv('data/election_results.csv')

In [114]:
election_results.head()

,year,prospective conscutive term for incumbent party,prospective 1st/2nd term for incumbent party candidate,incumbent party,popular vote for incumbent party,popular vote for non-incumbent party,winner party,winner president,loser
0,2024,1,1,Democratic,48.34,49.81,Republican,Donald Trump,Kamala Harris
1,2020,1,1,Republican,46.86,51.31,Democratic,Joe Biden,Donald Trump
2,2016,3,1,Democratic,48.20,46.20,Republican,Donald Trump,Hillary Clinton
3,2012,2,2,Democratic,51.10,47.20,Democratic,Barack Obama,Mitt Romney
4,2008,3,1,Republican,45.70,52.90,Democratic,Barack Obama,John McCain


In [115]:
prediction_data = pd.merge(election_results, gini, how = 'left', on=['year'])

In [116]:
prediction_data = pd.merge(prediction_data, gdp, how = 'left', on=['year'])

In [117]:
prediction_data = pd.merge(prediction_data, unemployment, how = 'left', on=['year'])

In [118]:
prediction_data = pd.merge(prediction_data, inflation, how = 'left', on=['year'])

In [119]:
yearly_wage['year'] = yearly_wage['year'].astype(int)
yearly_pers['year'] = yearly_pers['year'].astype(int)
yearly_us['year'] = yearly_us['year'].astype(int)

In [120]:
prediction_data = pd.merge(prediction_data, yearly_wage, how = 'left', on=['year'])
prediction_data = pd.merge(prediction_data, yearly_pers, how = 'left', on=['year'])
prediction_data = pd.merge(prediction_data, yearly_us, how = 'left', on=['year'])

In [121]:
prediction_data = prediction_data.dropna()

In [122]:
prediction_data

,year,prospective conscutive term for incumbent party,prospective 1st/2nd term for incumbent party candidate,incumbent party,popular vote for incumbent party,popular vote for non-incumbent party,winner party,winner president,loser,gini,gdp,unemployment,inflation,wage,pers,us
0,2024,1,1,Democratic,48.34,49.81,Republican,Donald Trump,Kamala Harris,41.8,1.794192,4.022,2.949525,369.50,78.0,20.0
1,2020,1,1,Republican,46.86,51.31,Democratic,Joe Biden,Donald Trump,40.0,-2.561807,8.055,1.233584,380.00,90.0,41.0
2,2016,3,1,Democratic,48.20,46.20,Republican,Donald Trump,Hillary Clinton,41.3,1.022666,4.869,1.261583,346.75,85.0,23.0
4,2008,3,1,Republican,45.70,52.90,Democratic,Barack Obama,John McCain,41.1,-0.828888,5.784,3.839100,335.25,80.0,10.0
5,2004,2,2,Republican,50.70,48.30,Republican,George W. Bush,John Kerry,40.5,2.891111,5.529,2.677237,337.50,84.0,45.0
6,2000,3,1,Democratic,48.40,47.90,Republican,George W. Bush,Al Gore,40.1,2.925863,3.992,3.376857,334.25,87.0,62.0
7,1996,2,2,Democratic,49.20,40.70,Democratic,Bill Clinton,Bob Dole,40.3,2.572464,5.451,2.931204,312.75,86.0,41.0
8,1992,4,2,Republican,37.40,43.00,Democratic,Bill Clinton,George H. W. Bush,38.4,2.096669,7.500,3.028820,313.50,78.0,22.5
9,1988,3,1,Republican,53.60,45.60,Republican,George H. W. Bush,Michael Dukakis,37.7,3.235338,5.500,4.077741,325.75,86.5,48.5
10,1984,2,2,Republican,58.80,40.60,Republican,Ronald Reagan,Walter Mondale,37.3,6.311989,7.500,4.300535,313.50,79.0,51.0


In [123]:
import statsmodels.api as sm

In [124]:
from ISLP import load_data
from ISLP.models import (ModelSpec as MS ,
summarize ,
poly)

In [141]:
X = prediction_data[['gini', 'gdp', 'unemployment', 'inflation', 'wage', 'pers', 'us', 'prospective conscutive term for incumbent party', 'prospective 1st/2nd term for incumbent party candidate']]

In [126]:
Y = prediction_data['popular vote for incumbent party']
model = sm.OLS(y, X)
results = model.fit()

In [87]:
results.summary()

c:\Users\carle\miniconda3\envs\datasci\lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=10 observations were given.
  return hypotest_fun_in(*args, **kwds)


<class 'statsmodels.iolib.summary.Summary'>
"""
                                        OLS Regression Results                                       
=====================================================================================================
Dep. Variable:     popular vote for incumbent party   R-squared (uncentered):                   0.996
Model:                                          OLS   Adj. R-squared (uncentered):              0.986
Method:                               Least Squares   F-statistic:                              99.51
Date:                              Fri, 15 May 2026   Prob (F-statistic):                     0.00152
Time:                                      13:52:21   Log-Likelihood:                         -25.848
No. Observations:                                10   AIC:                                      65.70
Df Residuals:                                     3   BIC:                                      67.81
Df Model:                                         7                                                  
Covariance Type:                          nonrobust                                                  
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
gini            -1.1936      2.019     -0.591      0.596      -7.618       5.231
gdp              1.9906      1.601      1.243      0.302      -3.104       7.085
unemployment    -0.3685      1.521     -0.242      0.824      -5.208       4.470
inflation        1.2293      2.399      0.512      0.644      -6.405       8.864
wage             0.1392      0.134      1.035      0.377      -0.289       0.567
pers             0.5585      0.837      0.667      0.552      -2.106       3.223
us              -0.0691      0.293     -0.236      0.829      -1.001       0.863
==============================================================================
Omnibus:                        5.036   Durbin-Watson:                   2.033
Prob(Omnibus):                  0.081   Jarque-Bera (JB):                1.932
Skew:                          -1.050   Prob(JB):                        0.381
Kurtosis:                       3.473   Cond. No.                         491.
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [85]:
summarize(results)

c:\Users\carle\miniconda3\envs\datasci\lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=10 observations were given.
  return hypotest_fun_in(*args, **kwds)


,coef,std err,t,P>|t|
gini,-1.1936,2.019,-0.591,0.596
gdp,1.9906,1.601,1.243,0.302
unemployment,-0.3685,1.521,-0.242,0.824
inflation,1.2293,2.399,0.512,0.644
wage,0.1392,0.134,1.035,0.377
pers,0.5585,0.837,0.667,0.552
us,-0.0691,0.293,-0.236,0.829


In [127]:
from functools import partial
from sklearn.model_selection import \
(cross_validate ,
KFold ,
ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

In [142]:
X = MS(['gini', 'gdp', 'unemployment', 'inflation', 'wage', 'pers', 'us', 'prospective conscutive term for incumbent party', 'prospective 1st/2nd term for incumbent party candidate']).fit_transform(prediction_data)
Y = prediction_data['popular vote for incumbent party']

model = sklearn_sm(sm.OLS, MS(['gini', 'gdp', 'unemployment', 'inflation', 'wage', 'pers', 'us', 'prospective conscutive term for incumbent party', 'prospective 1st/2nd term for incumbent party candidate']))
results = model.fit(X, Y)
predictions = model.predict(X)

cv_results = cross_validate(model ,
X,
Y,
cv = prediction_data.shape [0])
cv_err = np.mean(cv_results['test_score'])
cv_err

np.float64(2245.6353844692817)

In [135]:
X = MS(['gini', 'gdp', 'unemployment', 'inflation', 'wage', 'pers', 'us']).fit_transform(prediction_data)
Y = prediction_data['popular vote for incumbent party']

model = sklearn_sm(sm.OLS, MS(['gini', 'gdp', 'unemployment', 'inflation', 'wage', 'pers', 'us']))
results = model.fit(X, Y)
predictions = model.predict(X)

cv_results = cross_validate(model ,
X,
Y,
cv = prediction_data.shape [0])
cv_err = np.mean(cv_results['test_score'])
cv_err

np.float64(71.63698685915456)

In [139]:
X = MS(['gini', 'gdp', 'unemployment', 'inflation', 'wage']).fit_transform(prediction_data)
Y = prediction_data['popular vote for incumbent party']

model = sklearn_sm(sm.OLS, MS(['gini', 'gdp', 'unemployment', 'inflation', 'wage']))
results = model.fit(X, Y)
predictions = model.predict(X)

cv_results = cross_validate(model ,
X,
Y,
cv = prediction_data.shape [0])
cv_err = np.mean(cv_results['test_score'])
cv_err

np.float64(118.93052146366355)

In [140]:
X = MS(['pers', 'us']).fit_transform(prediction_data)
Y = prediction_data['popular vote for incumbent party']

model = sklearn_sm(sm.OLS, MS(['pers', 'us']))
results = model.fit(X, Y)
predictions = model.predict(X)

cv_results = cross_validate(model ,
X,
Y,
cv = prediction_data.shape [0])
cv_err = np.mean(cv_results['test_score'])
cv_err

np.float64(44.00889024604335)